In [ ]:
library(tidyverse)
library(janitor)
library(omicsTools)
library(PubChemR)
library(classyfireR)
library(VennDiagram)
library(grid)

In [ ]:
theme_function = function(){
 theme_classic() +
            theme(
                panel.grid.major.x = element_line(color = "gray80", linetype = "dashed"),
                panel.grid.major.y = element_line(color = "gray80", linetype = "dashed"),
                strip.text.y = element_text(angle = 0, size = 12),
                strip.text.x = element_text(angle = 0, size = 12),
                text = element_text(size = 12),
                axis.title.x = element_text(size = 12), 
                axis.title.y = element_text(size = 12),
                axis.text.x = element_text(size = 12), 
                axis.text.y = element_text(size = 12))
    }

In [ ]:
# QC control, remove impossible compounds
# siloxane and trimethylsilylated peaks

In [ ]:
removed = readxl::read_excel('./data/pdms_data/STable1_JMD Edits.xlsx', sheet = "Removed") %>% pull(Name) %>% unique()
length(removed)

In [ ]:
# Direct skin PDMS (not controls)
sample_id = as.character(c(1:4))
location_id = c("A", "B",'C', "D")
patterns = expand.grid(sample_id, location_id) %>% 
    mutate(pattern = paste0(Var1, Var2))
pattern_id = patterns$pattern

In [ ]:
datas= list()
for (i in pattern_id){
    out = tryCatch(
        expr = {
            readxl::read_excel('./data/pdms_data/Emory Skin Sample Data.xlsx', sheet = i) %>% 
                group_by(CAS) %>% 
                mutate(Area_sum = sum(Area)) %>%
                filter(!duplicated(CAS)) %>% 
                select(Name, CAS, !!i := Area_sum, Similarity, Reverse) %>% 
                # remove siloxane and trimethylsilylated peaks 
                filter(!Name %in% removed) %>% 
                # confident VOC  
                filter(Similarity >= 700 & Reverse >= 700 ) %>% 
                dplyr::select(!all_of(c("Similarity", "Reverse")))
        }, error = function(e){
            NULL
        }
        )
    datas[[i]] = out    
}

In [ ]:
# combine
data = reduce(datas, full_join, by = c("Name", "CAS")) 

In [ ]:
feature_ = data[, c(1,2)]
X = data[, c(-1, -2)]
dim(X)

In [ ]:
# remove call rate for features
n_samples = ncol(X)
prop_non_na = apply(!is.na(X), 1, sum)/n_samples

# keep at least m samples
m = 3
keep = prop_non_na > m/n_samples
sum(keep)

In [ ]:
X = X[keep, ]
feature_ = feature_[keep, ]
dim(X)

In [ ]:
data_ = bind_cols(feature_, log1p(X))
data_ %>% write_csv("./results/pdms_feature_filtered_normalized_ver2.csv")

In [ ]:
# remove call rate for samples
n_samples = ncol(X)
prop_non_na = apply(!is.na(X), 2, sum)/nrow(X)

# keep at least m features
m = 17.4
keep = prop_non_na > m/nrow(X)
print(m/nrow(X))
sum(keep)

In [ ]:
X = X[, keep]
dim(X)

In [ ]:
# total ion current (TIC) normalization is valid? 
# if many features, untargeted measurement, TIC can normalize technical sample variation.
# here, we have targeted GC-MS, and we expect some skin locations may have better/worse compound catchment, with sample variability (large).
# in this case, TIC may distort the signal.

# TIC
# X_rel = sweep(X, 2, colSums(X, na.rm = TRUE), FUN = "/")

In [ ]:
# simply log normalization
boxplot(log1p(X))

In [ ]:
data_selected = bind_cols(feature_, log1p(X))

In [ ]:
data_selected %>% write_csv("./results/pdms_filtered_normalized_ver2.csv")

In [ ]:
data = read_csv("./results/pdms_filtered_normalized_ver2.csv")

In [ ]:
print(dim(data))
data %>% pull(Name)

In [ ]:
# primary, realistic condition

In [ ]:
# compound heatmap
data_value = data %>% 
    select(!c(Name, CAS)) 
mat = data_value %>% as.matrix()

mat[is.na(mat)] = 0
dim(mat)
colnames(mat) = names(data_value)
mols = data %>% 
    pull(Name)
rownames(mat) = mols

In [ ]:
meta = tibble(sample_id = names(data_value)) %>% 
    mutate(donor_id = str_extract(sample_id, "\\d")) %>% 
    mutate(location = str_extract(sample_id, "[A-D]")) %>% 
    mutate(location = factor(location, 
                            levels = c('A', "B", "C", "D"), 
                            labels = c("Forehead", "Neck","Wrist", "Ankle"))) 

In [ ]:
anno_col = as.data.frame(meta)
rownames(anno_col) =anno_col$sample_id
anno_col = anno_col[,-1]
names(anno_col) = c("Donor ID", "Site")

In [ ]:
p = pheatmap::pheatmap(mat,  annotation_col = anno_col)$gtable
ggsave('./results/heatmap_primary_ver2.pdf', p, height = 24, width = 10, unit = "in", dpi = 400)

In [ ]:
D_bc <- vegan::vegdist(t(mat), method="bray", na.rm = FALSE)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 6, repr.plot.res = 100)

pco = cmdscale(D_bc, k = 2, eig = TRUE)
meta$sample_id

In [ ]:
df = data.frame(
  Sample = meta$sample_id,
  PCoA1 = pco$points[,1],
  PCoA2 = pco$points[,2],
  donor = meta$donor_id,
  site  = meta$location
)

In [ ]:
p = ggplot(df, aes(PCoA1, PCoA2, color = donor, shape = site)) +
  geom_point(size = 3) +
  theme_function() + 
  scale_color_discrete(name = "Donor", palette = "Set2") +
  scale_shape_discrete(name = "Site")

print(p)
ggsave('./results/pcoa_primary_ver2.pdf', p, height = 4, width = 6, unit = "in", dpi = 400)

In [ ]:
# RQ: donor variability
# if donor variablility can be detected, the variation should be larger than site variability.
# use beta diversity
# Bray–Curtis distance
D <- vegan::vegdist(t(mat), method = "bray")   # if X already log-transformed, use vegdist(X, ...)

dmat <- as.matrix(D)

don <- meta$donor_id

# 2) Extract upper triangle distances
within  <- dmat[outer(don, don, "==") & upper.tri(dmat)]
between <- dmat[outer(don, don, "!=") & upper.tri(dmat)]

# 3) Summaries + effect size
within_mean    = mean(within,  na.rm = TRUE)
between_mean   = mean(between, na.rm = TRUE)
diff = mean(between, na.rm = TRUE) - mean(within, na.rm = TRUE)

print(within_mean)
print(between_mean)
print(diff)

In [ ]:
don

In [ ]:
# permutation test
set.seed(1)

B <- 10000
perm <- replicate(B, {
  don_p <- sample(don)
  w <- dmat[outer(don_p, don_p, "==") & upper.tri(dmat)]
  b <- dmat[outer(don_p, don_p, "!=") & upper.tri(dmat)]
  mean(b, na.rm=TRUE) - mean(w, na.rm=TRUE)
})
pval <- mean(perm >= diff)
print(pval)

In [ ]:
# which location is credible
# use the all data (before removing samples)
data_ = read_csv("./results/pdms_feature_filtered_normalized_ver2.csv")
data_ = data_ %>% 
    pivot_longer(!c(Name, CAS), names_to = "sample_loc", values_to = "peak") %>% 
    mutate(donor_id = str_extract(sample_loc, "\\d")) %>% 
    mutate(location = str_extract(sample_loc, "[A-D]")) %>% 
    mutate(location = factor(location, 
                            levels = c('A', "B", "C", "D"), 
                            labels = c("Forehead", "Neck","Wrist", "Ankle"))) 

In [ ]:
n_features = data_ %>% distinct(Name) %>% pull(Name) %>% length(.)
data__ = data_ %>% 
    mutate(detected = ifelse(is.na(peak), 0, 1)) %>%
    group_by(sample_loc) %>% 
    mutate(n_detected = sum(detected)) %>% 
    ungroup() %>% 
    filter(!duplicated(sample_loc)) %>% 
    mutate(total = n_features) %>% 
    mutate(detected_prop = n_detected/total) 
p = data__ %>% 
    ggplot(aes(location, detected_prop)) + 
        geom_boxplot() + 
        geom_point(aes(color = donor_id), size= 3, position = position_dodge(width = 0.2)) + 
        theme_function() +
        scale_color_discrete(name = "Donor", palette = "Set2")+
        xlab("") + 
        ylab("Proportion (detected)")

print(p)
ggsave('./results/location_box_ver2.pdf', p, height = 4, width = 6, unit = "in", dpi = 400)

In [ ]:
data__ %>% write_csv("./results/detected_prop_ver2.csv")

In [ ]:
data__ = read_csv("./results/detected_prop_ver2.csv")

In [ ]:
data__ %>% 
    group_by(location) %>% 
    summarise(
        mean = mean(detected_prop),
        sd = sd(detected_prop),
        cv = sd/mean)

In [ ]:
# diagnosis of compounds comparing room, cloth, S3, empty GC-MS

In [ ]:
# forehead and ankle were not reliable, only using neck and wrist
data = read_csv("./results/pdms_feature_filtered_normalized_ver2.csv")
keep_cols = c(
    'Name', 'CAS', '2B','3B','4B','1C','2C','3C','4C')
data = data %>% dplyr::select(all_of(keep_cols)) 

# some CASs were incorrect
modify = tibble(
Name = c("2-(Diethylamino)acetonitrile", "Pentadecanal-","4-Vinyl-imidazole", "Ethyne, fluoro-"),
CAS = c("3010-02-4", "2765-11-9", "664107","2713-09-9")
)

for(i in 1:nrow(modify)){
    data[data$Name == modify$Name[i], "CAS"] = modify$CAS[i]
}    
skin_cas = data %>% pull(CAS)
length(skin_cas)

In [ ]:
# original one
skin_cas_original = read_csv("./results/skin_cas.csv") %>% pull(CAS)

In [ ]:
table(skin_cas %in% skin_cas_original)

In [ ]:
# clothing PDMS

In [ ]:
sheets = readxl::excel_sheets('./data/pdms_data/Emory Skin Sample Data.xlsx')
bools = sapply(sheets, function(x) str_detect(x, "E"))
pattern_id = sheets[bools]

In [ ]:
datas= list()
for (i in pattern_id){
    out = tryCatch(
        expr = {
            readxl::read_excel('./data/pdms_data/Emory Skin Sample Data.xlsx', sheet = i) %>% 
                group_by(CAS) %>% 
                mutate(Area_sum = sum(Area)) %>%
                filter(!duplicated(CAS)) %>% 
                select(Name, CAS, !!i := Area_sum, Similarity, Reverse) %>% 
                # remove siloxane and trimethylsilylated peaks 
                filter(!Name %in% removed)  %>% 
                # confident VOC  
                filter(Similarity >= 700 & Reverse >= 700 ) %>% 
                dplyr::select(!all_of(c("Similarity", "Reverse")))
        }, error = function(e){
            NULL
        }
        )
    datas[[i]] = out    
}


In [ ]:
# combine
combined = reduce(datas, full_join, by = c("Name", "CAS")) 

In [ ]:
combined %>% filter(!str_detect(CAS, "-")) %>% pull(Name)

modify = tibble(
Name = c(
'1,2-Propanediol, 2-acetate', 
'Cyclohexane, 1-ethenyl-1-methyl-2-(1-methylethenyl)-4-(1-methylethylidene)-',
'2-(Diethylamino)acetonitrile',
'1,3,5-Cycloheptatriene, 7,7-dimethyl-'),
CAS = c("6214-01-3", 
        "3242-08-8",
        "3010-02-4",
        "4082-89-7")
)

for(i in 1:nrow(modify)){
    combined[combined$Name == modify$Name[i], "CAS"] = modify$CAS[i]
}    

In [ ]:
combined %>% filter(!str_detect(CAS, "-")) %>% pull(Name)

In [ ]:
# cloth PDMS, under same conditions, cloth compounds should be consistent across samples, removing some contamination during handling
cloth_cas = combined %>% 
    mutate(credible = !is.na(`1E`) | !is.na(`2E`)) %>% 
    filter(credible) %>% 
    pull(CAS)
print(length(cloth_cas))

In [ ]:
# room control
sheets = readxl::excel_sheets('./data/pdms_data/Emory Skin Sample Data.xlsx')
bools = sapply(sheets, function(x) str_detect(x, "5"))
pattern_id = sheets[bools]
pattern_id               

In [ ]:
datas= list()
for (i in pattern_id){
    out = tryCatch(
        expr = {
            readxl::read_excel('./data/pdms_data/Emory Skin Sample Data.xlsx', sheet = i) %>% 
                group_by(CAS) %>% 
                mutate(Area_sum = sum(Area)) %>%
                filter(!duplicated(CAS)) %>% 
                select(Name, CAS, !!i := Area_sum, Similarity, Reverse) %>% 
                # remove siloxane and trimethylsilylated peaks 
                filter(!Name %in% removed)  %>% 
                # confident VOC  
                filter(Similarity >= 700 & Reverse >= 700 ) %>% 
                dplyr::select(!all_of(c("Similarity", "Reverse")))
        }, error = function(e){
            NULL
        }
        )
    datas[[i]] = out    
}


In [ ]:
# combine
combined = reduce(datas, full_join, by = c("Name", "CAS")) 
combined %>% filter(!str_detect(CAS, "-")) %>% pull(Name)

In [ ]:
modify = tibble(
Name = c(
'Methanesulfonic anhydride',
'1,2-Propanediol, 2-acetate',
'2-(Diethylamino)acetonitrile',
'1,3,5-Cycloheptatriene, 7,7-dimethyl-',
'Ethyne, fluoro-'),

CAS = c("7143-01-3", 
        "6214-01-3",
        "3010-02-4",
        "7557-11-1",
        "2713-09-9")
)

for(i in 1:nrow(modify)){
    combined[combined$Name == modify$Name[i], "CAS"] = modify$CAS[i]
}  

In [ ]:
combined %>% filter(!str_detect(CAS, "-")) %>% pull(Name)

In [ ]:
# env, under same conditions, env compounds should be consistent across samples, removing some contamination during handling
room_cas = combined %>% 
    mutate(credible = !is.na(`5A`) | !is.na(`5B`)) %>% 
    filter(credible) %>% 
    pull(CAS)
print(length(room_cas))

In [ ]:
# GC MS control
pattern_id = c("GCMS_only")

In [ ]:
datas= list()
for (i in pattern_id){
    out = tryCatch(
        expr = {
            readxl::read_excel('./data/pdms_data/Emory Skin Sample Data.xlsx', sheet = i) %>% 
                group_by(CAS) %>% 
                mutate(Area_sum = sum(Area)) %>%
                filter(!duplicated(CAS)) %>% 
                select(Name, CAS, !!i := Area_sum, Similarity, Reverse) %>% 
                # remove siloxane and trimethylsilylated peaks 
                filter(!Name %in% removed)  %>% 
                # confident VOC  
                filter(Similarity >= 700 & Reverse >= 700 ) %>% 
                dplyr::select(!all_of(c("Similarity", "Reverse")))
        }, error = function(e){
            NULL
        }
        )
    datas[[i]] = out    
}

In [ ]:
# combine
combined = reduce(datas, full_join, by = c("Name", "CAS")) 

In [ ]:
combined %>% filter(!str_detect(CAS, "-")) %>% pull(Name)

In [ ]:
# env, under same conditions, env compounds should be consistent across samples, removing some contamination during handling
gcms_only_cas = combined %>% 
    mutate(credible = !is.na(`GCMS_only`)) %>% 
    filter(credible) %>% 
    pull(CAS)
print(length(gcms_only_cas))

In [ ]:
# collect all compound and CAS ID
sheets = readxl::excel_sheets('./data/pdms_data/Emory Skin Sample Data.xlsx')
datas= list()
for (i in sheets){
    out = tryCatch(
        expr = {
            readxl::read_excel('./data/pdms_data/Emory Skin Sample Data.xlsx', sheet = i) %>% 
                group_by(CAS) %>% 
                mutate(Area_sum = sum(Area)) %>%
                filter(!duplicated(CAS)) %>% 
                select(Name, CAS, !!i := Area_sum, Similarity, Reverse) %>% 
                # remove siloxane and trimethylsilylated peaks 
                filter(!Name %in% removed)  %>% 
                # confident VOC  
                filter(Similarity >= 700 & Reverse >= 700 ) %>% 
                dplyr::select(!all_of(c("Similarity", "Reverse")))
                
        }, error = function(e){
            NULL
        }
        )
    datas[[i]] = out    
}

In [ ]:
cas_dict = reduce(datas, full_join, by = c("Name", "CAS")) %>% 
    select(Name, CAS)

In [ ]:
cas_dict %>% nrow()

In [ ]:
# sys review compounds -> cirpy to get CAS, but not complete
s3 = readxl::read_excel("./results/summary_cas.xlsx")
nrow(s3)

In [ ]:
s3 %>% dplyr::rename(name = Name) %>%
    mutate(name = str_to_lower(name)) %>% 
    left_join(cas_dict %>% mutate(name = str_to_lower(Name)), by = "name") %>% 
    writexl::write_xlsx("./results/summary_cas_added.xlsx")

In [ ]:
s3_modified = readxl::read_excel("./results/summary_cas_added_modified.xlsx")
s3_modified %>% nrow()

In [ ]:
# make sure to remove siloxane and trimethylsilylated peaks 
s3_modified = s3_modified %>% filter(!name %in% str_to_lower(removed))

In [ ]:
s3_modified %>% nrow()

In [ ]:
# keep only CAS available
s3_modified = s3_modified %>% filter(!is.na(CAS_new))
s3_cas = s3_modified %>% pull(CAS_new)
s3_modified %>% nrow()
length(s3_cas)

In [ ]:
# update cas_dict
s3_cas_dict = s3_modified %>% mutate(Name = str_to_title(name)) %>% 
    select(Name, CAS = CAS_new)

In [ ]:
cas_dict_new = bind_rows(cas_dict, s3_cas_dict) %>% 
    filter(!duplicated(CAS))

In [ ]:
cas_dict_new %>% write_csv("./results/cas_dict.csv")

In [ ]:
# some of CASs were incorrect. revise
modify1 = tibble(
Name = c("2-(Diethylamino)acetonitrile", "Pentadecanal-","4-Vinyl-imidazole", "Ethyne, fluoro-"),
CAS = c("3010-02-4", "2765-11-9", "664107","2713-09-9")
)

modify2 = tibble(
Name = c(
'1,2-Propanediol, 2-acetate', 
'Cyclohexane, 1-ethenyl-1-methyl-2-(1-methylethenyl)-4-(1-methylethylidene)-',
'2-(Diethylamino)acetonitrile',
'1,3,5-Cycloheptatriene, 7,7-dimethyl-'),
CAS = c("6214-01-3", 
        "3242-08-8",
        "3010-02-4",
        "4082-89-7")
)

modify3 = tibble(
Name = c(
'Methanesulfonic anhydride',
'1,2-Propanediol, 2-acetate',
'2-(Diethylamino)acetonitrile',
'1,3,5-Cycloheptatriene, 7,7-dimethyl-',
'Ethyne, fluoro-'),

CAS = c("7143-01-3", 
        "6214-01-3",
        "3010-02-4",
        "7557-11-1",
        "2713-09-9")
)

modify = bind_rows(modify1, modify2, modify3) %>% 
    filter(!duplicated(CAS))

In [ ]:
cas_dict = read_csv("./results/cas_dict.csv")

In [ ]:
# revise
for(i in 1:nrow(modify)){
    cas_dict[cas_dict$Name == modify$Name[i], "CAS"] = modify$CAS[i]
}  

In [ ]:
cas_dict %>% write_csv("./results/cas_dict_revised.csv")

In [ ]:
# now compare, detected skin, clothing, room, GCMS_only, Mitra

In [ ]:
# credible all compounts
all_cas = unique(c(skin_cas, cloth_cas, room_cas, gcms_only_cas, s3_cas))
length(all_cas)

In [ ]:
cas_dict_revised = read_csv("./results/cas_dict_revised.csv")

In [ ]:
detected_matrix = tibble(CAS = all_cas) %>% 
    mutate(skin =  ifelse(CAS %in% skin_cas, 1, 0)) %>% 
    mutate(cloth = ifelse(CAS %in% cloth_cas, 1, 0)) %>% 
    mutate(room = ifelse(CAS %in% room_cas, 1, 0)) %>% 
    mutate(gcms_only = ifelse(CAS %in% gcms_only_cas, 1, 0)) %>% 
    mutate(s3 = ifelse(CAS %in% s3_cas, 1, 0)) %>% 
    left_join(cas_dict_revised, by = "CAS") 

In [ ]:
detected_matrix %>% write_csv("./results/detected_matrix_ver2.csv")

In [ ]:
# Venn
detected_matrix = read_csv("./results/detected_matrix_ver2.csv")

In [ ]:
skin = detected_matrix %>% filter(skin == 1) %>% pull(CAS)
length(skin)

cloth = detected_matrix %>% filter(cloth == 1) %>% pull(CAS)
length(cloth)

room = detected_matrix %>% filter(room == 1) %>% pull(CAS)
length(room)

gcms_only = detected_matrix %>% filter(gcms_only == 1) %>% pull(CAS)
length(gcms_only)

s3 = detected_matrix %>% filter(s3 == 1) %>% pull(CAS)
length(s3)

In [ ]:
# up set plot
# row -> VOC
# columns - > catgory (binary)
data = detected_matrix %>% 
    dplyr::select(Name, 
                  skin, 
                  cloth, 
                  room, 
                  gcms_only, 
                  s3) %>% 
    dplyr::rename(Skin = skin, 
                  "Clothing-control" = cloth,
                 "Room-control" = room,
                 "Blank-control" = gcms_only, "Mitra et al." = s3) %>% 
    as.data.frame()

In [ ]:
library(ggplotify)
library(ComplexUpset)

In [ ]:
p = ComplexUpset::upset(
  data,
  intersect = rev(c(
    "Skin",
    "Clothing-control",
    "Room-control",
    "Blank-control",
    "Mitra et al."
  )),
  sort_sets = FALSE,
  sort_intersections = "descending",
  
    
  # top bar
  base_annotations = list(
   "VOC Intersections" =
    ComplexUpset::intersection_size() +
    scale_y_continuous(name = "VOC intersections") +
    theme(
      axis.title.y = element_text(size = 14),
      axis.text.y  = element_text(size = 12))),
      
  set_sizes = ComplexUpset::upset_set_size() +
     scale_y_continuous(name = "VOCs per experimental condition"),
  
 queries = list(
    ComplexUpset::upset_query(set = "Skin",color = "#E64B35", fill = "#E64B35"),
    ComplexUpset::upset_query(set = "Clothing-control",color = "#4DBBD5", fill = "#4DBBD5"),
    ComplexUpset::upset_query(set = "Room-control", color = "#00A087", fill = "#00A087"),
    ComplexUpset::upset_query(set = "Blank-control", color ="#8491B4", fill = "#8491B4"),
    ComplexUpset::upset_query(set = "Mitra et al.", color = "#F39B7F", fill = "#F39B7F")),
  themes = ComplexUpset::upset_modify_themes(
  list(

    # DOT MATRIX
    "intersections_matrix" = theme(
      axis.title.x = element_blank(),
      axis.text.y  = element_text(size = 13)
    ),

    # LEFT HORIZONTAL BAR PLOT
    "overall_sizes" = theme(
      axis.title.x = element_text(size = 14),
      axis.text.x  = element_text(size = 12)
    )
  )
))
    

ggsave("./results/upset_ver2.pdf", plot = as.ggplot(p), height = 5, width = 12, unit = "in", dpi = 400)
p

In [ ]:
library(gplots)

In [ ]:
x = list("Skin" = skin, 
         "Cloth" = cloth, 
         "Room" = room, 
         "Blank" =   gcms_only, 
         "Mitra et al" =   s3)

ItemsList <- venn(x, show.plot = FALSE)

lengths(attributes(ItemsList)$intersections)
list_res = attributes(ItemsList)$intersections

In [ ]:
# inspect which
cas_dict = read_csv("./results/cas_dict_revised.csv")

In [ ]:
list_res_renamed = list()
for(i in 1:length(list_res)){
    key = names(list_res)[i]
    values = list_res[[i]]
   
    values_re = cas_dict %>% filter(CAS %in% values) %>% pull(Name) %>% paste(collapse =", ")
    list_res_renamed[[key]] = values_re
}

In [ ]:
ns = lapply(list_res, function(x) length(x)) %>% unlist()

In [ ]:
tibble("Intersect" =  names(list_res),
      "N" = ns, 
      "VOC" = unlist(list_res_renamed)) %>% 
    writexl::write_xlsx('./results/table_intersect_ver2.xlsx')

In [ ]:
overlaps = readxl::read_excel('./results/table_intersect_ver2.xlsx')

In [ ]:
S = skin
C = cloth # Clothing PDMS
R = room # Room PDMS
E = gcms_only # Empty GCMS
M = s3 # Mit et al

In [ ]:
# In skin
length(S)

In [ ]:
# In skin, how many were clothing?
length(intersect(S, C))

In [ ]:
# In skin, how many were room control
length(intersect(S, R))

In [ ]:
# In skin, how many were blank control
length(intersect(S, E))

In [ ]:
# VOC overlapping across skin, cloth, room and blank
check = intersect(intersect(S, C),  intersect(R, E))
length(check)